In [1]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
import re
import xml.etree.ElementTree as ET
from functools import reduce
import openpyxl
from scipy.optimize import minimize_scalar
import numpy as np
import glob

pd.set_option('display.max_columns', None)

# Interest rate

In [2]:
INTEREST_URL = "https://index.minfin.com.ua/en/banks/nbu/refinance/"

In [ ]:
def scrape_nbu_rate_monthly_avg(start_month="2010-01", end_month="2025-12"):
    r = requests.get(INTEREST_URL, timeout=30)
    r.raise_for_status()

    soup = BeautifulSoup(r.text, "html.parser")
    text = soup.get_text("\n", strip=True)

    pattern = re.compile(
        r"from\s+(\d{2}\.\d{2}\.\d{4})"
        r"(?:\s+to\s+(\d{2}\.\d{2}\.\d{4}))?"
        r"\s+(\d+,\d{2})"
    )

    matches = pattern.findall(text)
    if not matches:
        raise ValueError("Could not parse the rate table. The page structure may have changed.")

    df = pd.DataFrame(matches, columns=["date_from", "date_to", "rate_raw"])

    df["date_from"] = pd.to_datetime(df["date_from"], format="%d.%m.%Y")
    df["date_to"] = pd.to_datetime(
        df["date_to"].replace("", pd.NA),
        format="%d.%m.%Y",
        errors="coerce"
    )
    df["rate"] = df["rate_raw"].str.replace(",", ".", regex=False).astype(float)

    df["date_to"] = df["date_to"].fillna(pd.Timestamp("2100-12-31"))

    df = df.sort_values("date_from").reset_index(drop=True)

    months = pd.period_range(start=start_month, end=end_month, freq="M")
    result = []

    for month in months:
        month_start = month.start_time.normalize()
        month_end = month.end_time.normalize()

        weighted_sum = 0.0
        total_days = 0

        for _, row in df.iterrows():
            overlap_start = max(month_start, row["date_from"])
            overlap_end = min(month_end, row["date_to"])

            if overlap_start <= overlap_end:
                days = (overlap_end - overlap_start).days + 1
                weighted_sum += row["rate"] * days
                total_days += days

        avg_rate = weighted_sum / total_days if total_days > 0 else pd.NA

        result.append({
            "month": str(month),
            "value": avg_rate
        })

    out = pd.DataFrame(result)
    return out

In [4]:
monthly_avg_rates = scrape_nbu_rate_monthly_avg()
monthly_avg_rates["value"] = monthly_avg_rates["value"].round(4)

monthly_avg_rates.tail()

,month,value
187,2025-08,15.5
188,2025-09,15.5
189,2025-10,15.5
190,2025-11,15.5
191,2025-12,15.5


In [5]:
monthly_avg_rates["month"] = pd.to_datetime(monthly_avg_rates["month"])
monthly_avg_rates["year-quarter"] = monthly_avg_rates["month"].dt.to_period("Q").astype(str)

quarterly_rates = (
    monthly_avg_rates.groupby("year-quarter", as_index=False)["value"]
    .mean()
    .rename(columns={"value": "interest_rate_avg_q"})
)

quarterly_rates

,year-quarter,interest_rate_avg_q
0,2010Q1,10.250000
1,2010Q2,10.058333
2,2010Q3,8.147833
3,2010Q4,7.750000
4,2011Q1,7.750000
...,...,...
59,2024Q4,13.102167
60,2025Q1,14.521533
61,2025Q2,15.500000
62,2025Q3,15.500000


# Exchange rate

In [6]:
def fetch_nbu_fx_quarterly_avg(valcode="USD", start="2010-01-01", end="2024-12-31"):
    url = "https://bank.gov.ua/NBU_Exchange/exchange_site"

    params = {
        "start": pd.to_datetime(start).strftime("%Y%m%d"),
        "end": pd.to_datetime(end).strftime("%Y%m%d"),
        "valcode": valcode,
        "json": "",
        "sort": "exchangedate",
        "order": "asc",
    }

    r = requests.get(url, params=params, timeout=60)
    r.raise_for_status()
    data = r.json()

    df = pd.DataFrame(data)
    if df.empty:
        return df

    df["dt"] = pd.to_datetime(df["exchangedate"], format="%d.%m.%Y")
    df["year-quarter"] = df["dt"].dt.to_period("Q").astype(str)

    out = (
        df.groupby("year-quarter", as_index=False)["rate"]
        .mean()
        .rename(columns={"rate": f"{valcode.lower()}_avg_q"})
    )

    return out

In [7]:
usd_q = fetch_nbu_fx_quarterly_avg(
    valcode="USD",
    start="2010-01-01",
    end="2024-12-31"
)

eur_q = fetch_nbu_fx_quarterly_avg(
    valcode="EUR",
    start="2010-01-01",
    end="2024-12-31"
)

exchange_rates = pd.merge(usd_q, eur_q, on="year-quarter", how="outer").sort_values("year-quarter").reset_index(drop=True)
exchange_rates.head()

,year-quarter,usd_avg_q,eur_avg_q
0,2010Q1,798.770556,1107.207786
1,2010Q2,792.243077,1010.101635
2,2010Q3,790.062717,1017.562051
3,2010Q4,793.136848,1078.512101
4,2011Q1,794.495222,1084.949968


# CPI

In [8]:
CPI_URL = 'https://stat.gov.ua/sdmx/workspaces/default:integration/registry/sdmx/3.0/data/dataflow/SSSU/DF_PRICE_CHANGE_CONSUMER_GOODS_SERVICE/27.0.0/INDEX_CONSUMPRICE.DEC2010.UA00000000000000000.0.M'

def fetch_sdmx_series1525(url: str) -> pd.DataFrame:
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/122.0.0.0 Safari/537.36"
        ),
        "Accept": "application/xml, text/xml, */*",
        "Referer": "https://stat.gov.ua/uk/development-api/sdmx-api-v3",
    }

    response = requests.get(url, headers=headers, timeout=60)
    response.raise_for_status()

    root = ET.fromstring(response.content)

    series = root.find(".//Series")
    if series is None:
        raise ValueError("No <Series> found in response")

    scale = pd.to_numeric(series.attrib.get("SCALE"), errors="coerce")

    rows = []
    for obs in series.findall("./Obs"):
        rows.append({
            "year_quarter": obs.attrib.get("TIME_PERIOD"),
            "value": pd.to_numeric(obs.attrib.get("OBS_VALUE"), errors="coerce"),
            "scale": scale
        })

    df = pd.DataFrame(rows)

    if df.empty:
        raise ValueError("Series found, but no observations were extracted")

    return df

In [9]:
cpi_base_2010 = fetch_sdmx_series1525(CPI_URL)[["year_quarter", "value"]]

cpi_base_2010 = cpi_base_2010.drop(columns=["scale"], errors="ignore")

cpi_base_2010["date"] = pd.PeriodIndex(
    cpi_base_2010["year_quarter"].str.replace("-M", "-", regex=False),
    freq="M"
)

cpi_base_2010 = (
    cpi_base_2010.groupby(cpi_base_2010["date"].dt.asfreq("Q"))["value"]
    .mean()
    .reset_index()
)

cpi_base_2010["year_quarter"] = cpi_base_2010["date"].astype(str)
cpi_base_2010 = cpi_base_2010.drop(columns="date")
cpi_base_2010 = cpi_base_2010.rename(columns={"value": "cpi_base_2010"})
cpi_base_2010 = cpi_base_2010[
    (cpi_base_2010["year_quarter"] < "2025Q1") &
    (cpi_base_2010["year_quarter"] >= "2010Q1")
].reset_index(drop=True)
cpi_base_2010 = cpi_base_2010[["year_quarter", "cpi_base_2010"]]

cpi_base_2010

,year_quarter,cpi_base_2010
0,2010Q1,94.766667
1,2010Q2,95.166667
2,2010Q3,96.166667
3,2010Q4,99.366667
4,2011Q1,102.066667
5,2011Q2,105.366667
6,2011Q3,104.300000
7,2011Q4,104.400000
8,2012Q1,105.033333
9,2012Q2,105.000000


# Final

In [10]:
rates = exchange_rates.merge(quarterly_rates, on="year-quarter", how="outer").sort_values("year-quarter").reset_index(drop=True) \
    .merge(cpi_base_2010, left_on="year-quarter", right_on="year_quarter", how="outer") \
    .drop(columns=["year-quarter"], errors="ignore")
rates = rates[rates['year_quarter'] < '2025Q1']

rates = rates[["year_quarter", "usd_avg_q", "eur_avg_q", "interest_rate_avg_q", "cpi_base_2010"]]
rates

,year_quarter,usd_avg_q,eur_avg_q,interest_rate_avg_q,cpi_base_2010
0,2010Q1,798.770556,1107.207786,10.250000,94.766667
1,2010Q2,792.243077,1010.101635,10.058333,95.166667
2,2010Q3,790.062717,1017.562051,8.147833,96.166667
3,2010Q4,793.136848,1078.512101,7.750000,99.366667
4,2011Q1,794.495222,1084.949968,7.750000,102.066667
5,2011Q2,797.097253,1147.687592,7.750000,105.366667
6,2011Q3,797.170652,1127.782843,7.750000,104.300000
7,2011Q4,798.276957,1076.499504,7.750000,104.400000
8,2012Q1,798.815495,1045.743513,7.725800,105.033333
9,2012Q2,799.000769,1026.279065,7.500000,105.000000


In [11]:
rates["year_quarter"] = pd.PeriodIndex(rates["year_quarter"], freq="Q")

mask = rates["year_quarter"] < pd.Period("2020Q1", freq="Q")

rates.loc[mask, ["usd_avg_q", "eur_avg_q"]] = (
    rates.loc[mask, ["usd_avg_q", "eur_avg_q"]] / 100
)

In [13]:
rates.to_csv("data/rates_2010_2024.csv", index=False)